# Breast Tumor CT Segmentation — Kaggle GPU Training

Research prototype — **not a clinical diagnostic system**. Trains the same
U-Net segmentation pipeline from the GitHub repo:
[https://github.com/ahmad-pervaiz/Breast-Cancer-Detection-3D-Model](https://github.com/ahmad-pervaiz/Breast-Cancer-Detection-3D-Model)

**Before running:** Settings (right panel) →
- Accelerator: **GPU T4 x2** (or P100)
- Internet: **On** (needed to `git clone` the repo)
- Add Data → attach your uploaded `breast_tumor_dataset.zip` Kaggle Dataset

See `WALKTHROUGH.md` in the repo for the full step-by-step guide.


## 1. Environment check — confirm GPU is actually attached

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| CUDA version:", torch.version.cuda)
else:
    print("\n*** WARNING: no GPU detected. ***")
    print("Go to Settings (right panel) -> Accelerator -> GPU T4 x2 (or P100), then Save and re-run this cell.")
    print("Training will proceed on CPU if you continue, but will take hours instead of minutes.")


## 2. Clone the project code from GitHub

If you didn't push to GitHub yet, upload this project folder as a second Kaggle Dataset instead and adjust  below.

In [ ]:
import os, subprocess, shutil

REPO_URL = "https://github.com/ahmad-pervaiz/Breast-Cancer-Detection-3D-Model.git"
# The GitHub repo root IS this project (configs/, src/, scripts/ live at repo
# root) - there is no nested bc_tumor_detection/ subfolder inside the clone.
PROJECT_DIR = "/kaggle/working/Breast-Cancer-Detection-3D-Model"

if not os.path.exists(os.path.join(PROJECT_DIR, "configs", "config.yaml")):
    # Missing or incomplete (e.g. a previous clone attempt failed partway) - wipe and retry.
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

if not os.path.exists(os.path.join(PROJECT_DIR, "configs", "config.yaml")):
    raise RuntimeError(
        f"Clone finished but {PROJECT_DIR}/configs/config.yaml is still missing - "
        f"check the repo actually contains this project at its root: {REPO_URL}"
    )

os.chdir(PROJECT_DIR)
print("cwd:", os.getcwd())


In [ ]:
!ls

## 3. Install the few dependencies Kaggle's base image doesn't already have

In [ ]:
!pip install -q albumentations opencv-python-headless pyyaml
print("done")


## 4. Locate the attached dataset

Kaggle mounts every attached Dataset under `/kaggle/input/<dataset-slug>/`.
This cell finds it automatically — it looks for the one containing a `Train/`
folder (the FINAL DATASET layout), so it works regardless of what you named
the dataset at upload time.

In [ ]:
from pathlib import Path

REQUIRED_SUBDIRS = ["Train", "Train_masks", "valid", "valid_masks", "test"]
candidates = [
    p for p in Path("/kaggle/input").iterdir()
    if p.is_dir() and all((p / sub).exists() for sub in REQUIRED_SUBDIRS)
]
if not candidates:
    raise RuntimeError(
        "No attached dataset with Train/Train_masks/valid/valid_masks/test folders found under "
        "/kaggle/input. Add Data -> attach your breast_tumor_dataset.zip Kaggle Dataset, then re-run this cell."
    )
if len(candidates) > 1:
    print(f"WARNING: multiple matching datasets found, using the first: {candidates}")
DATASET_ROOT = str(candidates[0])
print("Using dataset_root:", DATASET_ROOT)
!ls "{DATASET_ROOT}"


## 5. Dataset audit

Same integrity checks that ran locally (corruption, mask binarity, label
consistency, patient-leakage assertion) — re-verified here since this is a
fresh environment with a fresh copy of the data.

In [ ]:
!python scripts/audit_dataset.py --config configs/config.yaml --dataset_root "{DATASET_ROOT}"


## 6. Smoke test

2 epochs on a tiny stratified subset — confirms the pipeline runs correctly
in this environment (GPU detected, data loads, checkpoints save) before
committing GPU-hours to a full run. Should take well under a minute.

In [ ]:
!python scripts/train.py --config configs/config.yaml --dataset_root "{DATASET_ROOT}" --smoke_test


## 7. Full training

Uses every setting from `configs/config.yaml` (loss weights, augmentation,
early stopping, etc.) except `dataset_root`, which is overridden to the
Kaggle mount path. Edit `--epochs`/`--batch_size` here if you want to deviate
from the config defaults for this run.

In [ ]:
!python scripts/train.py --config configs/config.yaml --dataset_root "{DATASET_ROOT}" --epochs 50 --batch_size 16


## 8. View training curves + a validation prediction montage inline

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for name in ["loss_curve.png", "dice_curve.png", "iou_curve.png"]:
    img = Image.open(f"runs/segmentation/plots/{name}")
    plt.figure(figsize=(6, 4)); plt.imshow(img); plt.axis("off"); plt.title(name); plt.show()


In [ ]:
import glob
latest_montage = sorted(glob.glob("runs/segmentation/predictions/validation/epoch_*.png"))[-1]
Image.open(latest_montage)


## 9. Zip results for download

Kaggle auto-saves everything under `/kaggle/working/` as the notebook's
output, but a single zip of just the checkpoints/logs/plots is easier to
grab from the Output tab than digging through the repo clone.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/segmentation_results", "zip", "runs/segmentation")
print("Saved: /kaggle/working/segmentation_results.zip")
print("Download it from the notebook's Output tab (top right, after committing/saving the run).")


## 10. Next steps

1. Download `segmentation_results.zip` from the Output tab.
2. Unzip it locally into `bc_tumor_detection/runs/segmentation/` (matching
   the layout the repo expects).
3. Run inference locally (or in another Kaggle cell) with:

```
python scripts/inference.py \
    --input "FINAL DATASET/test" \
    --checkpoint runs/segmentation/checkpoints/best_model.pth \
    --output runs/segmentation/test_predictions/
```

See `WALKTHROUGH.md` for the full guide, including how to interpret the
metrics and what NOT to do with the test set.
